# ASTRA-LM Experiment Analysis
Use this notebook to analyze and compare results from different experiment runs.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import glob
from IPython.display import display

## Select Experiment Directory
Change the `run_path` to point to your experiment folder (e.g., `outputs/compare_gpt_vs_vayusphere/2026...`).

In [ ]:
run_path = "outputs/compare_gpt_vs_vayusphere/" # Update this

all_runs = sorted(glob.glob(os.path.join(run_path, "*")))
if all_runs:
    latest_run = all_runs[-1]
    print(f"Latest run: {latest_run}")
    run_to_analyze = latest_run
else:
    print("No runs found in", run_path)
    run_to_analyze = None

## Load and Plot Metrics

In [ ]:
if run_to_analyze:
    subdirs = [d for d in os.listdir(run_to_analyze) if os.path.isdir(os.path.join(run_to_analyze, d))]
    
    plt.figure(figsize=(12, 6))
    
    for subdir in subdirs:
        metrics_file = os.path.join(run_to_analyze, subdir, "metrics.csv")
        if os.path.exists(metrics_file):
            df = pd.read_csv(metrics_file)
            # Plot loss
            plt.plot(df['step'], df['loss'].interpolate(), label=f"{subdir} - Train Loss", alpha=0.3)
            
            # Plot eval loss
            eval_df = df.dropna(subset=['eval_loss'])
            if not eval_df.empty:
                plt.plot(eval_df['step'], eval_df['eval_loss'], marker='o', label=f"{subdir} - Eval Loss")
    
    plt.title(f"Loss Comparison: {os.path.basename(run_to_analyze)}")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.show()

## VayuSphere Diagnostics
If VayuSphere was active, we can check its gate and gradient statistics.

In [ ]:
if run_to_analyze:
    for subdir in subdirs:
        metrics_file = os.path.join(run_to_analyze, subdir, "metrics.csv")
        if os.path.exists(metrics_file):
            df = pd.read_csv(metrics_file)
            if 'vs_q_gate_mean' in df and not df['vs_q_gate_mean'].dropna().empty:
                print(f"\n--- VayuSphere Diagnostics for {subdir} ---")
                display(df[['step', 'vs_q_gate_mean', 'vs_k_gate_mean', 'vs_centroid_grad_norm_mean']].dropna().tail())
                
                plt.figure(figsize=(10, 4))
                plt.plot(df['step'], df['vs_q_gate_mean'].interpolate(), label='Q Gate Mean')
                plt.plot(df['step'], df['vs_k_gate_mean'].interpolate(), label='K Gate Mean')
                plt.title(f"VayuSphere Gates: {subdir}")
                plt.legend()
                plt.show()